# EDA 03: The district table (ethanol first)  *(TODO: Sage)*

**EDA steps 5–6.** One row per district (75, keyed on `lgd_district_code`), one column per
variable that might go into the ethanol cost model, with its coverage shown. Then: which
variables are redundant (step 6)?

| | |
|---|---|
| **Inputs** | the spine; the loaders' tidy tables in `data/interim/` (EDA 10 `apy`, EDA 11 `climate`, more as downloads arrive); the ethanol profile (`load_profile`) |
| **Outputs** | `data/interim/district/district_table_ethanol.parquet`; a coverage table per column; findings in `docs/eda/03_district_table.md` |
| **Rule** | Start from the spine and **left join** everything onto it. Every section ends with the same invariant: 75 rows, unique LGD codes. A missing value is a finding, not a reason to drop a district |

Sections 5 and 6 wait for the manual downloads (groundwater, land); their cells say so.
Learn: `docs/learn/duckdb.md`, `docs/eda/plan.md` (the variable list and why).

In [ ]:
# Remove the `noqa: F401` mark once you use duckdb.
import duckdb  # noqa: F401
import pandas as pd

from up_siting.common.paths import INTERIM_DIR, PRODUCT_PROFILES_DIR, SEEDS_DIR
from up_siting.profiles.validate import load_profile

spine = pd.read_csv(SEEDS_DIR / "lgd_district_spine.csv")
APY = INTERIM_DIR / "apy" / "apy_up_long.parquet"
POWER = INTERIM_DIR / "climate" / "power_daily.parquet"
HQ = INTERIM_DIR / "climate" / "hq_points.parquet"
OUT_DIR = INTERIM_DIR / "district"

# Public values only. To use the partner's figures, pass PRODUCT_PROFILES_DIR / "private"
# as the second argument, and DON'T commit outputs that print them.
ethanol = load_profile(PRODUCT_PROFILES_DIR / "ethanol.yaml")


def check_table(df: pd.DataFrame) -> None:
    # The invariant every section re-checks.
    assert len(df) == 75, len(df)
    assert df["lgd_district_code"].is_unique
    assert set(df["lgd_district_code"]) == set(spine["lgd_district_code"])


district = spine[["lgd_district_code", "district_name", "in_census_2011"]].copy()
check_table(district)

## 1. Raw material volume: cane, maize, rice

Question: how much of each ethanol feedstock does each district produce, recently and on
average, and how steady is it?

Watch DQ-007: cane is reported for 45 districts from 2020-21, with a break in the series.

In [ ]:
# TODO(sage)
# 1. From APY (UPAG rows from 2013, DES before), annual production per district and crop
#    (sum the seasons) for sugarcane, maize, rice.
# 2. Per district and crop:
#      <crop>_t_recent   mean of the last 3 crop years available (2022-2024)
#      <crop>_cv_10y     coefficient of variation over 10 years; for cane use 2020-2024 only
#                        (or flag it), never across the 2020-21 break
#      <crop>_trend_pct  optional: slope of a linear fit, % per year
# 3. cane_recent_reported = True if the district reports cane in the latest year.
# 4. Left join onto `district` (pivot to one column per crop/measure first).
volume: pd.DataFrame | None = None

In [ ]:
assert isinstance(volume, pd.DataFrame)
district_v = district.merge(volume, on="lgd_district_code", how="left", validate="one_to_one")
check_table(district_v)
assert district_v["cane_recent_reported"].sum() in (44, 45)  # DQ-007
assert (district_v["maize_t_recent"].fillna(0) >= 0).all()

## 2. Feedstock indicators per route

Question: per ethanol route, how many tonnes of usable feedstock does each district have?
Uses the profile's `availability` (`from_variable`, `ratio`). Existing distillery use
(`dfpd_distilleries`) is subtracted once that source is loaded.

In [ ]:
# TODO(sage)
# 1. For each raw material in ethanol.raw_materials with an availability.ratio:
#      indicator_<id>_t = <from_variable>_t_recent x ratio
#    (sugarcane_crushed isn't loaded yet: use sugarcane production as its proxy, S-1, and
#    say so in a column note / the findings.)
# 2. Leave rice's indicator null if its ratio is null (the profile says why).
# 3. Later: minus existing use from dfpd_distilleries (KLPD x days x input ratio).
indicators: pd.DataFrame | None = None

In [ ]:
assert isinstance(indicators, pd.DataFrame)
check_table(district.merge(indicators, on="lgd_district_code", how="left"))
assert "indicator_c_heavy_molasses_t" in indicators.columns

## 3. Climate flags

Question: which districts are hotter, drier or more erratic? (Context and risk flags, not a
cost term: GHI barely varies, DQ-009.)

In [ ]:
# TODO(sage)
# From POWER (2001-2025) per district: rain_mm_per_year (mean of annual sums),
# rain_cv (std/mean of annual sums), hot_days_ge_40c (mean per year), ghi_kwh_m2_day (mean).
# Also carry hq_points.point_kind so a fallback point is visible.
climate: pd.DataFrame | None = None

In [ ]:
assert isinstance(climate, pd.DataFrame)
check_table(district.merge(climate, on="lgd_district_code", how="left"))
assert climate["rain_cv"].between(0, 1).all()

## 4. Prices (statewide constants for now)

Cane SAP and the FCI rice price are statewide: put them in as constants from the profile
(with their provenance), not as columns that vary. Maize prices by district arrive with
`agmarknet_ceda` (card D-E9b): Census 2011 ids → `spine.census_2011_code`; the four
post-2011 districts take their parent's value (`district_lineage.csv`), flagged.

In [ ]:
# TODO(sage) after the CEDA download: maize_price_inr_per_t (median of monthly modal prices,
# last 24 months) per district, with a `maize_price_from_parent` flag.
prices: pd.DataFrame | None = None

## 5. Water: share of stressed groundwater blocks  *(waits for `cgwb_gw`)*

Question: what share of each district's blocks are over-exploited or critical?

In [ ]:
# TODO(sage) after the CGWB download and its loader:
#   gw_share_oe_critical = (# blocks over-exploited + critical) / (# blocks), per district
#   gw_n_blocks; and the worst category present.
water: pd.DataFrame | None = None

## 6. Land  *(waits for the circle-rate sample and industrial rates)*

In [ ]:
# TODO(sage) after the downloads: land_rural_inr_per_sqm (sample districts only; others null),
# industrial_park_inr_per_sqm (where a park exists). Coverage will be partial by design (S-2).
land: pd.DataFrame | None = None

## 7. Join everything and show coverage per column

Question: for each column, how many of the 75 districts have a value? (The keep rule needs
≥ 70.)

In [ ]:
# TODO(sage)
# 1. district_table = district, left-joined with every frame above that isn't None
#    (validate="one_to_one" on each merge).
# 2. coverage = one row per column: n_non_null, share, passes_70 (n_non_null >= 70).
district_table: pd.DataFrame | None = None
coverage: pd.DataFrame | None = None

In [ ]:
assert isinstance(district_table, pd.DataFrame) and isinstance(coverage, pd.DataFrame)
check_table(district_table)
assert set(coverage.columns) >= {"column", "n_non_null", "share", "passes_70"}
assert not district_table.columns.duplicated().any()

## 8. Redundancy (EDA step 6)

Question: which variables measure the same thing (e.g. cane volume and the molasses
indicator are proportional by construction)? Spearman correlations between numeric
columns; flag pairs with |ρ| > 0.8 and decide which one stays.

In [ ]:
# TODO(sage)
# corr = district_table.select_dtypes("number").drop(columns="lgd_district_code").corr("spearman")
# redundant = pairs with |rho| > 0.8 (upper triangle only), sorted.
redundant: pd.DataFrame | None = None

## 9. Write and read back

In [ ]:
# TODO(sage): OUT_DIR.mkdir; write district_table_ethanol.parquet; read back with DuckDB;
# assert 75 rows. Then write the findings below and in docs/eda/03_district_table.md.

In [ ]:
assert (OUT_DIR / "district_table_ethanol.parquet").is_file()

## 10. Findings  *(TODO: Sage)*

1. *Columns that pass the keep rule (≥ 70 districts, recent, varies) …*
2. *Redundant pairs and which one stays …*
3. *Districts with the most gaps, and why …*